In [36]:
from folium.plugins import HeatMapWithTime

def build_heatmap_with_time(df):
    # Prepare data for HeatMapWithTime: one frame per (day, hour)
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    hours = list(range(24))
    frames = []
    time_labels = []
    for day in days:
        for hour in hours:
            subset = df[(df["Incident Day of Week"] == day) & (df["hour"] == hour)]
            points = subset[["Latitude", "Longitude"]].dropna().values.tolist()
            frames.append(points)
            time_labels.append(f"{day} {hour:02d}:00")

    m = folium.Map(location=[37.7749, -122.4194], zoom_start=12.4, tiles="CartoDB Positron", control_scale=True)
    HeatMapWithTime(
        frames,
        index=time_labels,
        auto_play=True,
        max_opacity=0.7,
        radius=16,
        gradient={0.2: "#b9d7d9", 0.5: "#4d8b99", 1.0: "#d04f2d"},
        use_local_extrema=True,
    ).add_to(m)
    m.save(str(ASSET_DIR / "night_map_with_time.html"))
    print(f"Wrote {ASSET_DIR / 'night_map_with_time.html'}")
{
    "cells": [
        {
            "cell_type": "markdown",
            "id": "#VSC-174fdec7",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "# Build Time Story",
                "",
                "This notebook rebuilds the Week 8 article assets inside `Lecture 8`.",
                "",
                "It generates:",
                "- `assets/daily_rhythm.png`",
                "- `assets/night_map.html`",
                "- `data/time_story.json`",
                "- `index.html`",
                "",
                "The web page itself is styled by `styles.css` and powered by `script.js`.",
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-94d85a60",
            "metadata": {
                "language": "python"
            },
            "source": [
                "from pathlib import Path",
                "import json",
                "",
                "import folium",
                "import matplotlib",
                "matplotlib.use(\"Agg\")",
                "import matplotlib.pyplot as plt",
                "import numpy as np",
                "import pandas as pd",
                "import seaborn as sns",
                "from folium.plugins import HeatMap",
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-acd3dcea",
            "metadata": {
                "language": "python"
            },
            "source": [
                "ROOT = Path.cwd()",
                "if ROOT.name != \"Lecture 8\":",
                "    ROOT = ROOT / \"Lecture 8\"",
                "",
                "DATA_PATH = ROOT.parent / \"Lecture 2\" / \"SF_Crime_Merged_FINAL_CLEAN_2003_2026.csv\"",
                "ASSET_DIR = ROOT / \"assets\"",
                "DATA_DIR = ROOT / \"data\"",
                "HTML_PATH = ROOT / \"index.html\"",
                "JSON_PATH = DATA_DIR / \"time_story.json\"",
                "STATIC_FIG_PATH = ASSET_DIR / \"daily_rhythm.png\"",
                "MAP_PATH = ASSET_DIR / \"night_map.html\"",
                "",
                "FOCUS_CATEGORIES = [",
                "    \"Vehicle Theft\",",
                "    \"Robbery\",",
                "    \"Drug Offense\",",
                "    \"Warrants\",",
                "    \"Weapon Offense\",",
                "    \"Missing Person\",",
                "    \"Non-Criminal\",",
                "]",
                "",
                "WEEKDAY_ORDER = [",
                "    \"Monday\",",
                "    \"Tuesday\",",
                "    \"Wednesday\",",
                "    \"Thursday\",",
                "    \"Friday\",",
                "    \"Saturday\",",
                "    \"Sunday\",",
                "]",
                "",
                "PALETTE = {",
                "    \"weekday\": \"#245c69\",",
                "    \"weekend\": \"#d04f2d\",",
                "    \"ink\": \"#18261f\",",
                "    \"muted\": \"#5e675f\",",
                "    \"paper\": \"#f6efe3\",",
                "    \"grid\": \"#d7d0c4\",",
                "}",
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-5bb85ec1",
            "metadata": {
                "language": "python"
            },
            "source": [
                "def build_static_chart(df):",
                "    unique_dates = df[[\"date\", \"is_weekend\"]].drop_duplicates()",
                "    date_counts = unique_dates.groupby(\"is_weekend\").size().to_dict()",
                "",
                "    hourly = (",
                "        df.groupby([\"is_weekend\", \"hour\"])",
                "        .size()",
                "        .rename(\"count\")",
                "        .reset_index()",
                "    )",
                "    hourly[\"avg_per_day\"] = hourly.apply(lambda row: row[\"count\"] / date_counts[row[\"is_weekend\"]], axis=1)",
                "",
                "    weekday_curve = hourly[hourly[\"is_weekend\"] == False].sort_values(\"hour\")",
                "    weekend_curve = hourly[hourly[\"is_weekend\"] == True].sort_values(\"hour\")",
                "    diff = weekend_curve[\"avg_per_day\"].to_numpy() - weekday_curve[\"avg_per_day\"].to_numpy()",
                "",
                "    sns.set_theme(style=\"whitegrid\")",
                "    plt.rcParams[\"font.family\"] = \"DejaVu Sans\"",
                "",
                "    fig = plt.figure(figsize=(16, 11), facecolor=PALETTE[\"paper\"])",
                "    gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.1], hspace=0.22)",
                "    ax1 = fig.add_subplot(gs[0, 0], facecolor=PALETTE[\"paper\"])",
                "    ax2 = fig.add_subplot(gs[1, 0], facecolor=PALETTE[\"paper\"], sharex=ax1)",
                "",
                "    for ax in (ax1, ax2):",
                "        ax.spines[[\"top\", \"right\"]].set_visible(False)",
                "        ax.grid(color=PALETTE[\"grid\"], alpha=0.35, linewidth=0.7)",
                "        ax.set_axisbelow(True)",
                "",
                "    hours = weekday_curve[\"hour\"].to_numpy()",
                "    weekday_vals = weekday_curve[\"avg_per_day\"].to_numpy()",
                "    weekend_vals = weekend_curve[\"avg_per_day\"].to_numpy()",
                "",
                "    ax1.plot(hours, weekday_vals, color=PALETTE[\"weekday\"], linewidth=2.5, label=\"Average weekday\")",
                "    ax1.plot(hours, weekend_vals, color=PALETTE[\"weekend\"], linewidth=2.5, label=\"Average weekend day\")",
                "    ax1.fill_between(hours, weekend_vals, weekday_vals, where=weekend_vals >= weekday_vals, color=PALETTE[\"weekend\"], alpha=0.10)",
                "    ax1.fill_between(hours, weekend_vals, weekday_vals, where=weekend_vals < weekday_vals, color=PALETTE[\"weekday\"], alpha=0.06)",
                "    ax1.set_ylim(2, 26)",
                "",
                "    # Reduce number of annotation arrows and use clearer placement",
                "    ax1.annotate(",
                "        \"Weekend nights pull ahead after midnight\",",
                "        xy=(1, weekend_vals[1]),",
                "        xytext=(2.1, 22.9),",
                "        arrowprops={\"arrowstyle\": \"-\", \"lw\": 1.2, \"color\": PALETTE[\"weekend\"]},",
                "        fontsize=12,",
                "        color=PALETTE[\"ink\"],",
                "        ha=\"left\",",
                "        va=\"top\"",
                "    )",
                "    ax1.annotate(",
                "        \"Weekdays dominate the workday and early evening\",",
                "        xy=(17, weekday_vals[17]),",
                "        xytext=(11.2, 21.6),",
                "        arrowprops={\"arrowstyle\": \"-\", \"lw\": 1.2, \"color\": PALETTE[\"weekday\"]},",
                "        fontsize=12,",
                "        color=PALETTE[\"ink\"],",
                "        ha=\"left\",",
                "        va=\"top\"",
                "    )",
                "",
                "    ax1.set_title(\"Crime follows office hours until midnight flips the script\", fontsize=24, loc=\"left\", pad=22, color=PALETTE[\"ink\"], weight=\"bold\")",
                "    ax1.text(0, 1.01, \"Average incidents per hour on a weekday versus an average weekend day, 2003-2025.\", transform=ax1.transAxes, fontsize=13, color=PALETTE[\"muted\"])",
                "    ax1.set_ylabel(\"Average incidents per day\", fontsize=16, color=PALETTE[\"ink\"])",
                "    ax1.legend(frameon=False, ncol=2, loc=\"upper right\", fontsize=12)",
                "    ax1.tick_params(axis='both', which='major', labelsize=13)",
                "",
                "    ax2.axhline(0, color=PALETTE[\"muted\"], linewidth=1)",
                "    ax2.fill_between(hours, diff, 0, where=diff >= 0, color=PALETTE[\"weekend\"], alpha=0.55)",
                "    ax2.fill_between(hours, diff, 0, where=diff < 0, color=PALETTE[\"weekday\"], alpha=0.55)",
                "    ax2.set_ylabel(\"Weekend - weekday\", fontsize=14, color=PALETTE[\"ink\"])",
                "    ax2.set_xlabel(\"Hour of day\", fontsize=15, color=PALETTE[\"ink\"])",
                "    ax2.set_xticks(range(0, 24, 2))",
                "    ax2.tick_params(axis='both', which='major', labelsize=12)",
                "",
                "    fig.text(",
                "        0.125,",
                "        0.012,",
                "        \"The weekday curve is higher for most of the daytime because San Francisco has more weekday activity overall. But on a per-day basis, the weekend overtakes weekdays after dark and remains stronger around midnight, showing that incidents are not randomly distributed through the clock.\",",
                "        fontsize=13,",
                "        color=PALETTE[\"muted\"],",
                "    )",
                "",
                "    print(f\"Saving static chart to: {STATIC_FIG_PATH.resolve()}\")",
                "    fig.savefig(STATIC_FIG_PATH, dpi=200, bbox_inches=\"tight\", facecolor=fig.get_facecolor())",
                "",
                "    plt.close(fig)"
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-00656683",
            "metadata": {
                "language": "python"
            },
            "source": [
                "ASSET_DIR.mkdir(exist_ok=True)",
                "DATA_DIR.mkdir(exist_ok=True)",
                "",
                "usecols = [",
                "    \"datetime\",",
                "    \"Incident Day of Week\",",
                "    \"hour\",",
                "    \"Latitude\",",
                "    \"Longitude\",",
                "    \"Standardized Category FINAL\",",
                "    \"year\",",
                "]",
                "",
                "df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=[\"datetime\"])",
                "df = df.rename(columns={\"Standardized Category FINAL\": \"category\"})",
                "df = df[df[\"year\"].between(2003, 2025)].copy()",
                "df = df[df[\"hour\"].notna() & df[\"datetime\"].notna()].copy()",
                "df[\"hour\"] = df[\"hour\"].astype(int)",
                "df[\"date\"] = df[\"datetime\"].dt.date",
                "df[\"Incident Day of Week\"] = pd.Categorical(df[\"Incident Day of Week\"], categories=WEEKDAY_ORDER, ordered=True)",
                "df[\"is_weekend\"] = df[\"Incident Day of Week\"].isin([\"Saturday\", \"Sunday\"])",
                "",
                "build_static_chart(df)",
                "build_json(df)",
                "",
                "geo_df = df[df[\"Latitude\"].notna() & df[\"Longitude\"].notna()].copy()",
                "geo_df = geo_df[geo_df[\"Latitude\"].between(37.70, 37.83) & geo_df[\"Longitude\"].between(-122.53, -122.35)].copy()",
                "build_map(geo_df)",
                "",
                "print(f\"Wrote {STATIC_FIG_PATH}\")",
                "print(f\"Wrote {MAP_PATH}\")",
                "print(f\"Wrote {JSON_PATH}\")",
                ""
            ]
        },
        {
            "cell_type": "markdown",
            "id": "#VSC-88e66ef3",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## Final Page",
                "",
                "After running the build cells above, open `Lecture 8/index.html` in a browser. The page uses exactly three visualizations:",
                "1. A static chart",
                "2. A map",
                "3. An interactive Plotly chart",
                ""
            ]
        }
    ]
}

{'cells': [{'cell_type': 'markdown',
   'id': '#VSC-174fdec7',
   'metadata': {'language': 'markdown'},
   'source': ['# Build Time Story',
    '',
    'This notebook rebuilds the Week 8 article assets inside `Lecture 8`.',
    '',
    'It generates:',
    '- `assets/daily_rhythm.png`',
    '- `assets/night_map.html`',
    '- `data/time_story.json`',
    '- `index.html`',
    '',
    'The web page itself is styled by `styles.css` and powered by `script.js`.',
    '']},
  {'cell_type': 'code',
   'id': '#VSC-94d85a60',
   'metadata': {'language': 'python'},
   'source': ['from pathlib import Path',
    'import json',
    '',
    'import folium',
    'import matplotlib',
    'matplotlib.use("Agg")',
    'import matplotlib.pyplot as plt',
    'import numpy as np',
    'import pandas as pd',
    'import seaborn as sns',
    'from folium.plugins import HeatMap',
    '']},
  {'cell_type': 'code',
   'id': '#VSC-acd3dcea',
   'metadata': {'language': 'python'},
   'source': ['ROOT = Path.cwd(

# Build Time Story

This notebook rebuilds the Week 8 article assets inside `Lecture 8`.

It generates:
- `assets/daily_rhythm.png`
- `assets/night_map.html`
- `data/time_story.json`
- `index.html`

The web page itself is styled by `styles.css` and powered by `script.js`.


In [37]:
from pathlib import Path
import json

import folium
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from folium.plugins import HeatMap


In [38]:
ROOT = Path.cwd()
if ROOT.name != "Lecture 8":
    ROOT = ROOT / "Lecture 8"

DATA_PATH = ROOT.parent / "Lecture 2" / "SF_Crime_Merged_FINAL_CLEAN_2003_2026.csv"
ASSET_DIR = ROOT / "assets"
DATA_DIR = ROOT / "data"
HTML_PATH = ROOT / "index.html"
JSON_PATH = DATA_DIR / "time_story.json"
STATIC_FIG_PATH = ASSET_DIR / "daily_rhythm.png"
MAP_PATH = ASSET_DIR / "night_map.html"

FOCUS_CATEGORIES = [
    "Vehicle Theft",
    "Robbery",
    "Drug Offense",
    "Warrants",
    "Weapon Offense",
    "Missing Person",
    "Non-Criminal",
]

WEEKDAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

PALETTE = {
    "weekday": "#245c69",
    "weekend": "#d04f2d",
    "ink": "#18261f",
    "muted": "#5e675f",
    "paper": "#f6efe3",
    "grid": "#d7d0c4",
}


In [39]:
def build_static_chart(df):
    unique_dates = df[["date", "is_weekend"]].drop_duplicates()
    date_counts = unique_dates.groupby("is_weekend").size().to_dict()

    hourly = (
        df.groupby(["is_weekend", "hour"])
        .size()
        .rename("count")
        .reset_index()
    )
    hourly["avg_per_day"] = hourly.apply(lambda row: row["count"] / date_counts[row["is_weekend"]], axis=1)

    weekday_curve = hourly[hourly["is_weekend"] == False].sort_values("hour")
    weekend_curve = hourly[hourly["is_weekend"] == True].sort_values("hour")
    diff = weekend_curve["avg_per_day"].to_numpy() - weekday_curve["avg_per_day"].to_numpy()

    sns.set_theme(style="whitegrid")
    plt.rcParams["font.family"] = "DejaVu Sans"

    fig = plt.figure(figsize=(16, 11), facecolor=PALETTE["paper"])
    gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.1], hspace=0.22)
    ax1 = fig.add_subplot(gs[0, 0], facecolor=PALETTE["paper"])
    ax2 = fig.add_subplot(gs[1, 0], facecolor=PALETTE["paper"], sharex=ax1)

    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(color=PALETTE["grid"], alpha=0.35, linewidth=0.7)
        ax.set_axisbelow(True)

    hours = weekday_curve["hour"].to_numpy()
    weekday_vals = weekday_curve["avg_per_day"].to_numpy()
    weekend_vals = weekend_curve["avg_per_day"].to_numpy()

    ax1.plot(hours, weekday_vals, color=PALETTE["weekday"], linewidth=2.5, label="Average weekday")
    ax1.plot(hours, weekend_vals, color=PALETTE["weekend"], linewidth=2.5, label="Average weekend day")
    ax1.fill_between(hours, weekend_vals, weekday_vals, where=weekend_vals >= weekday_vals, color=PALETTE["weekend"], alpha=0.10)
    ax1.fill_between(hours, weekend_vals, weekday_vals, where=weekend_vals < weekday_vals, color=PALETTE["weekday"], alpha=0.06)
    ax1.set_ylim(2, 26)

    # Reduce number of annotation arrows and use clearer placement
    ax1.annotate(
        "Weekend nights pull ahead after midnight",
        xy=(1, weekend_vals[1]),
        xytext=(2.1, 22.9),
        arrowprops={"arrowstyle": "-", "lw": 1.2, "color": PALETTE["weekend"]},
        fontsize=12,
        color=PALETTE["ink"],
        ha="left",
        va="top"
    )
    ax1.annotate(
        "Weekdays dominate the workday and early evening",
        xy=(17, weekday_vals[17]),
        xytext=(11.2, 21.6),
        arrowprops={"arrowstyle": "-", "lw": 1.2, "color": PALETTE["weekday"]},
        fontsize=12,
        color=PALETTE["ink"],
        ha="left",
        va="top"
    )

    ax1.set_title("Crime follows office hours until midnight flips the script", fontsize=24, loc="left", pad=22, color=PALETTE["ink"], weight="bold")
    ax1.text(0, 1.01, "Average incidents per hour on a weekday versus an average weekend day, 2003-2025.", transform=ax1.transAxes, fontsize=13, color=PALETTE["muted"])
    ax1.set_ylabel("Average incidents per day", fontsize=16, color=PALETTE["ink"])
    ax1.legend(frameon=False, ncol=2, loc="upper right", fontsize=12)
    ax1.tick_params(axis='both', which='major', labelsize=13)

    ax2.axhline(0, color=PALETTE["muted"], linewidth=1)
    ax2.fill_between(hours, diff, 0, where=diff >= 0, color=PALETTE["weekend"], alpha=0.55)
    ax2.fill_between(hours, diff, 0, where=diff < 0, color=PALETTE["weekday"], alpha=0.55)
    ax2.set_ylabel("Weekend - weekday", fontsize=14, color=PALETTE["ink"])
    ax2.set_xlabel("Hour of day", fontsize=15, color=PALETTE["ink"])
    ax2.set_xticks(range(0, 24, 2))
    ax2.tick_params(axis='both', which='major', labelsize=12)

    fig.text(
        0.125,
        0.012,
        "The weekday curve is higher for most of the daytime because San Francisco has more weekday activity overall. But on a per-day basis, the weekend overtakes weekdays after dark and remains stronger around midnight, showing that incidents are not randomly distributed through the clock.",
        fontsize=13,
        color=PALETTE["muted"],
    )

    print(f"Saving static chart to: {STATIC_FIG_PATH.resolve()}")
    fig.savefig(STATIC_FIG_PATH, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

    plt.close(fig)

In [40]:
ASSET_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

usecols = [
    "datetime",
    "Incident Day of Week",
    "hour",
    "Latitude",
    "Longitude",
    "Standardized Category FINAL",
    "year",
]

df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=["datetime"])
df = df.rename(columns={"Standardized Category FINAL": "category"})
df = df[df["year"].between(2003, 2025)].copy()
df = df[df["hour"].notna() & df["datetime"].notna()].copy()
df["hour"] = df["hour"].astype(int)
df["date"] = df["datetime"].dt.date
df["Incident Day of Week"] = pd.Categorical(df["Incident Day of Week"], categories=WEEKDAY_ORDER, ordered=True)
df["is_weekend"] = df["Incident Day of Week"].isin(["Saturday", "Sunday"])

build_static_chart(df)
build_json(df)

geo_df = df[df["Latitude"].notna() & df["Longitude"].notna()].copy()
geo_df = geo_df[geo_df["Latitude"].between(37.70, 37.83) & geo_df["Longitude"].between(-122.53, -122.35)].copy()
build_map(geo_df)

print(f"Wrote {STATIC_FIG_PATH}")
print(f"Wrote {MAP_PATH}")
print(f"Wrote {JSON_PATH}")


Saving static chart to: D:\DTU\Social data analysis and Interaction\code\Lecture 8\assets\daily_rhythm.png
Wrote d:\DTU\Social data analysis and Interaction\code\Lecture 8\assets\daily_rhythm.png
Wrote d:\DTU\Social data analysis and Interaction\code\Lecture 8\assets\night_map.html
Wrote d:\DTU\Social data analysis and Interaction\code\Lecture 8\data\time_story.json


## Final Page

After running the build cells above, open `Lecture 8/index.html` in a browser. The page uses exactly three visualizations:
1. A static chart
2. A map
3. An interactive Plotly chart
